<a href="https://colab.research.google.com/github/hannahchen63/veritasai/blob/main/notebooks/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd

URL = "https://raw.githubusercontent.com/jabenitez88/NLP-EatingDisordersBERT/main/tweets_labelled_2000.csv"
ed = pd.read_csv(URL, sep=";")
ed = ed[["text_orig", "ProED"]].rename(columns={"text_orig": "text", "ProED": "label"})
print(ed.shape)
print(ed["label"].value_counts())

(2000, 2)
label
0    1533
1     467
Name: count, dtype: int64


Cell #1: Load the data

Load the NLP-EatingDisordersBERT dataset from github

Keep two columns: the text and the label

For labels, 1 = promotes eating disorder, 0 = does not

In [13]:
from sklearn.model_selection import train_test_split
import re

def clean(text):
    text = re.sub(r"^RT\s+", "", text)
    text = re.sub(r"@\w+", "@user", text)
    text = re.sub(r"http\S+", "", text)
    return re.sub(r"\s+", " ", text).strip()

ed = ed.dropna()
ed["text"] = ed["text"].astype(str).map(clean)

train, temp = train_test_split(ed, test_size=0.3, stratify=ed["label"], random_state=42)
val, test   = train_test_split(temp, test_size=0.5, stratify=temp["label"], random_state=42)
print(len(train), len(val), len(test))   # 1400 300 300

1400 300 300


Cell #2: Split the data

Split the dataset into training, validation, and testing

Stratify keeps about 23% of harmful tweets in each group

Random_state=42 makes the split the same everytime you run it

In [14]:
from sklearn.metrics import confusion_matrix

def report(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"{name}")
    print(f"  missed harmful posts (false negatives): {fn}/{fn+tp}")
    print(f"  harmless posts blurred (false positives): {fp}/{fp+tn}")
    print(f"  accuracy: {(tp+tn)/len(y_true):.2f}\n")

Cell #3: Report predictions

Reports predictions to their true labels

Calculates false negatives (missing harmful posts)

Calculates false positives (harmless posts get blurred)

In [15]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. dumb baselines
report("Always show", test.label, [0]*len(test))
report("Always blur", test.label, [1]*len(test))

# 2. keyword filter: what a simple non-AI extension would do
KEYWORDS = ["calorie", "calories", "kcal", "diet", "fasting", "fast",
            "skinny", "thin", "weight", "fat", "lose", "starve"]
def keyword_filter(text):
    return int(any(re.search(rf"\b{k}\b", text.lower()) for k in KEYWORDS))
report("Keyword filter", test.label, test.text.map(keyword_filter))

# 3. classic ML: TF-IDF + logistic regression
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(vec.fit_transform(train.text), train.label)
report("TF-IDF + LogReg", test.label, clf.predict(vec.transform(test.text)))

Always show
  missed harmful posts (false negatives): 70/70
  harmless posts blurred (false positives): 0/230
  accuracy: 0.77

Always blur
  missed harmful posts (false negatives): 0/70
  harmless posts blurred (false positives): 230/230
  accuracy: 0.23

Keyword filter
  missed harmful posts (false negatives): 49/70
  harmless posts blurred (false positives): 25/230
  accuracy: 0.75

TF-IDF + LogReg
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 13/230
  accuracy: 0.93



Cell #4: Baselines

Baselines for our AI to beat

Always show and always blur are extremes

The keyword filter only blurs a test if it has a certain word like "calories" or "diet"

TF-IDF + LogReg is a classic ML that turns tweets into weighted word counts and learns which words lean harmful (does not learn context of the full tweet)

In [16]:
!pip install -q datasets
from datasets import load_dataset

rec = load_dataset("AWeirdDev/all-recipes-xs", split="train").to_pandas()
rec["text"] = (rec["name"].astype(str) + ". " + rec["review"].astype(str)).str.slice(0, 280)
rec = rec[["text"]].drop_duplicates()
rec["label"] = 0
print(len(rec), "recipes")
print(rec.text.head(3).tolist())   # check they look reasonable

# 400 recipes go into training, the rest are held out as a recipe test set
rec_train = rec.sample(400, random_state=42)
rec_test  = rec.drop(rec_train.index)

train_mixed = pd.concat([train, rec_train], ignore_index=True).sample(frac=1, random_state=42)
print(train_mixed.label.value_counts())

500 recipes
['Dutch Oven Crunchy Corned Beef. I found this recipe attached to a 6-pack of beer on St. Patrick\'s Day in 1995.\nI\'ve been serving it as a holiday tradition since. Even self-proclaimed corned beef haters love this corned beef. A typical comment is, "could be the best corned beef I\'', 'Spicy Spicy Ranch Dressing. This spicy ranch dip with hot sauce and cayenne is excellent served with fries, poppers, and even wings.', 'Amazing Oven-Roasted Potatoes. These oven-roasted potatoes are golden on the outside and tender on the inside. Made with minimum oil for a less greasy option than regular fried potatoes!']
label
0    1473
1     327
Name: count, dtype: int64


Cell #5: Load recipes and build mixed dataset

Downloads AWeirdDev/all-recipes-xs dataset from HuggingFace

Combines recipe's name and short description into one text, under 280 characters

Labels every recipe as 0 (harmless)

Splits the recipe with 400 in training and the rest as a test set the model never sees

Adds the 400 training to the original training tweets to make a mixed dataset (train_mixed) and shuffles it

In [17]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)   # should say cuda

MODEL_NAME = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_loader(df, shuffle):
    enc = tokenizer(list(df.text), truncation=True, padding="max_length",
                    max_length=128, return_tensors="pt")
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"],
                       torch.tensor(df.label.values))
    return DataLoader(ds, batch_size=16, shuffle=shuffle)

train_loader = make_loader(train_mixed, True)
rec_test_loader = make_loader(rec_test, False)
print("Training on", len(train_loader.dataset), "examples")
val_loader   = make_loader(val, False)
test_loader  = make_loader(test, False)

Using: cuda
Training on 1800 examples


Cell #6: Setting up real model

Loads DistilRoBERTa-base model

Tokenizer turns tweets into numbers

DataLoader gives tweets in batch size 16

cuda is colab gpu

In [18]:
torch.manual_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# class weights: harmful posts are rarer (about 1 in 4), so missing one costs more
n0, n1 = (train_mixed.label == 0).sum(), (train_mixed.label == 1).sum()
weights = torch.tensor([1.0, n0 / n1], dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

def get_probs(loader):
    model.eval(); probs = []
    with torch.no_grad():
        for ids, mask, _ in loader:
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.array(probs)

for epoch in range(4):
    model.train(); total = 0
    for ids, mask, labels in train_loader:
        optimizer.zero_grad()
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        loss = loss_fn(logits, labels.to(device))
        loss.backward(); optimizer.step(); total += loss.item()
    val_pred = (get_probs(val_loader) >= 0.5).astype(int)
    print(f"Epoch {epoch+1}: train loss {total/len(train_loader):.3f}")
    report(f"  validation", val.label, val_pred)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1: train loss 0.306
  validation
  missed harmful posts (false negatives): 10/70
  harmless posts blurred (false positives): 15/230
  accuracy: 0.92

Epoch 2: train loss 0.154
  validation
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 21/230
  accuracy: 0.90

Epoch 3: train loss 0.111
  validation
  missed harmful posts (false negatives): 10/70
  harmless posts blurred (false positives): 11/230
  accuracy: 0.93

Epoch 4: train loss 0.060
  validation
  missed harmful posts (false negatives): 10/70
  harmless posts blurred (false positives): 15/230
  accuracy: 0.92



Cell #7: Training (fine-tuning)

Adds "harmful or not" layer to the model

Trains 1,400 tweets with 4 epochs (4 passes through the data)

Class weights means that harmful posts are rare (1 out of 4) so making a mistake on them costs 3x more to make the model not lazy

Loss goes down from 0.345 to 0.080

After each epoch, it checks the validation set

In [19]:
val_probs = get_probs(val_loader)
print("threshold | missed harmful (FN) | harmless blurred (FP)")
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(val.label, p, labels=[0, 1]).ravel()
    print(f"   {t:.1f}    |        {fn}/{fn+tp}         |      {fp}/{fp+tn}")

threshold | missed harmful (FN) | harmless blurred (FP)
   0.2    |        8/70         |      19/230
   0.3    |        9/70         |      17/230
   0.4    |        10/70         |      16/230
   0.5    |        10/70         |      15/230
   0.6    |        10/70         |      14/230
   0.7    |        11/70         |      12/230


Cell #8: Checking threshold

Model outputs score from 0 to 1 for "how harmful" text is

Threshold decides cutoff for blurring

Low threshold = blur more, fewer harmful posts get through, but more harmless posts get blurred wrongly

In [20]:
THRESHOLD = 0.4

In [21]:
test_pred = (get_probs(test_loader) >= THRESHOLD).astype(int)
report(f"DistilRoBERTa (threshold {THRESHOLD})", test.label, test_pred)

DistilRoBERTa (threshold 0.4)
  missed harmful posts (false negatives): 6/70
  harmless posts blurred (false positives): 19/230
  accuracy: 0.92



Cell #9: Final test

Runs trained model on test data set it has never seen

Results shown

Run 1, without cleaning twitter features:

DistilRoBERTa (threshold 0.4)
  missed harmful posts (false negatives): 7/70
  harmless posts blurred (false positives): 12/230
  accuracy: 0.94

Run 2, with cleaning twitter only (no recipes yet)

DistilRoBERTa (threshold 0.4)
  missed harmful posts (false negatives): 9/70
  harmless posts blurred (false positives): 14/230
  accuracy: 0.92

In [22]:
test_pred = (get_probs(test_loader) >= THRESHOLD).astype(int)
report("Mixed model - original test", test.label, test_pred)

rec_pred = (get_probs(rec_test_loader) >= THRESHOLD).astype(int)
print(f"Mixed model - recipes wrongly blurred: {rec_pred.sum()}/{len(rec_test)}")

# keyword filter and TF-IDF on the same recipes, for comparison
print(f"Keyword filter - recipes blurred: {rec_test.text.map(keyword_filter).sum()}/{len(rec_test)}")
print(f"TF-IDF - recipes blurred: {clf.predict(vec.transform(rec_test.text)).sum()}/{len(rec_test)}")

Mixed model - original test
  missed harmful posts (false negatives): 6/70
  harmless posts blurred (false positives): 19/230
  accuracy: 0.92

Mixed model - recipes wrongly blurred: 0/100
Keyword filter - recipes blurred: 2/100
TF-IDF - recipes blurred: 0/100


Cell #10: Compare

Runs new model on original set to make sure adding recipes does not harm the detection

Counts how many recipes the the new model wrongly blurs

Runs keyword filter and TF-IDF on the same recipes for comparison